In [1]:
!pip install -q peft bitsandbytes faiss-cpu sentence-transformers
!pip install -q evaluate rouge-score bert-score sacrebleu tqdm
!pip install -q langchain-community langchain-huggingface langchain-text-splitters langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requ

In [2]:
import os
import gc
import json
import torch
import warnings
import logging
import pandas as pd  # Kaggle Pandas nguyên bản!
from tqdm.notebook import tqdm 

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.getLogger("transformers").setLevel(logging.ERROR)

def flush_memory():
    """Hàm dọn VRAM cứu rỗi Kaggle"""
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

from datasets import load_dataset
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers.trainer_callback import PrinterCallback
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
from peft import LoraConfig, prepare_model_for_kbit_training, PeftModel, get_peft_model
import evaluate

In [3]:
class Config:
    # 1. Đường dẫn file dữ liệu thực tế của bạn
    KB_PATH = "/kaggle/input/datasets/phantrihieu/school-rules-and-regulations/data/nb/chunked_knowledge_base.jsonl"
    TRAIN_QA_PATH = "/kaggle/input/datasets/phantrihieu/school-rules-and-regulations/data/qa/train_qa_v2.jsonl"
    TEST_QA_PATH = "/kaggle/input/datasets/phantrihieu/school-rules-and-regulations/data/qa/test_qa_v2.jsonl"
    
    # 2. RAG Parameters
    EMBEDDING_MODEL = "keepitreal/vietnamese-sbert"
    TOP_K = 3 # top_k lúc tạo context trả lời câu hỏi (khi tính recall thì sẽ truyền riêng k=5)
    VECTOR_STORE_PATH = "/kaggle/working/faiss_index"
    
    # 3. LLM & Fine-tuning Parameters
    MODEL_ID = "SeaLLMs/SeaLLM-7B-v2"
    OUTPUT_DIR = "/kaggle/working/finetuned_model"
    MAX_STEPS = 200 
    LEARNING_RATE = 2e-4
    BATCH_SIZE = 2

In [4]:
class DataManager:
    @staticmethod
    def load_qa_data(file_path):
        print(f"⏳ Đang tải dữ liệu từ {file_path}...")
        # Lấy trực tiếp split train để tương thích với HF Trainer
        dataset = load_dataset("json", data_files={"train": file_path})['train']
        return dataset

class RAGPipeline:
    def __init__(self, config):
        self.config = config
        self.embedding = HuggingFaceEmbeddings(model_name=self.config.EMBEDDING_MODEL)
        self.vector_store = None

    def build_vector_store(self):
        print(f"⏳ Đang đọc Knowledge Base từ {self.config.KB_PATH}...")
        docs = []
        with open(self.config.KB_PATH, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    item = json.loads(line)
                    text_content = f"Tiêu đề: {item.get('title', '')}\nNội dung: {item.get('content', '')}"
                    metadata = {"source": item.get("source", ""), "source_url": item.get("source_url", "")}
                    docs.append(Document(page_content=text_content, metadata=metadata))
        
        print(f"⏳ Đang nhúng {len(docs)} chunks vào FAISS...")
        for _ in tqdm(range(1), desc="Building FAISS Index"):
            self.vector_store = FAISS.from_documents(documents=docs, embedding=self.embedding)
            
        self.vector_store.save_local(self.config.VECTOR_STORE_PATH)
        print("✅ Đã xây dựng và lưu Vector Store thành công!")

    def retrieve(self, query, top_k=None):
        if not self.vector_store:
            return "", []
        k = top_k if top_k else self.config.TOP_K
        docs = self.vector_store.similarity_search(query, k=k)
        return "\n\n".join([doc.page_content for doc in docs]), docs

In [5]:
class LLMManager:
    def __init__(self, config):
        self.config = config
        print(f"⏳ Đang tải Tokenizer và Base Model: [{self.config.MODEL_ID}] (4-bit)...")
        
        self.tokenizer = AutoTokenizer.from_pretrained(self.config.MODEL_ID)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16
        )
        
        self.model = AutoModelForCausalLM.from_pretrained(
            self.config.MODEL_ID, quantization_config=bnb_config, device_map="auto"
        )
        self.model = prepare_model_for_kbit_training(self.model)
        print(f"✅ Load Base Model [{self.config.MODEL_ID}] hoàn tất!")

    def finetune(self, train_dataset):
        peft_config = LoraConfig(
            r=16, lora_alpha=32, target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
            lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
        )
        self.model = get_peft_model(self.model, peft_config)

        def tokenize_function(examples):
            prompts = [f"### Câu hỏi:\n{q}\n\n### Trả lời:\n{a}" for q, a in zip(examples['instruction'], examples['output'])]
            return self.tokenizer(prompts, truncation=True, max_length=512)
            
        print("⏳ Đang xử lý Tokenize dữ liệu...")
        tokenized_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=train_dataset.column_names)

        training_args = TrainingArguments(
            output_dir=self.config.OUTPUT_DIR,
            per_device_train_batch_size=self.config.BATCH_SIZE,
            gradient_accumulation_steps=4, optim="paged_adamw_32bit",
            save_steps=50, logging_steps=10, learning_rate=self.config.LEARNING_RATE,
            fp16=True, max_steps=self.config.MAX_STEPS, report_to="none"
        )
        
        trainer = Trainer(
            model=self.model,
            train_dataset=tokenized_dataset,
            args=training_args,
            data_collator=DataCollatorForLanguageModeling(tokenizer=self.tokenizer, mlm=False)
        )
        
        # Xóa tính năng in dictionary xấu xí ra màn hình
        trainer.remove_callback(PrinterCallback)
        
        print("\n" + "🌟"*30)
        print(f"🚀 BẮT ĐẦU FINE-TUNING MÔ HÌNH: {self.config.MODEL_ID}")
        print("🌟"*30)
        
        trainer.train()
        trainer.model.save_pretrained(self.config.OUTPUT_DIR)
        print(f"✅ Fine-tuning hoàn tất! Đã lưu Adapter tại: {self.config.OUTPUT_DIR}")

    def load_adapter(self, adapter_path):
        print(f"⏳ Đang gắn Adapter từ {adapter_path} vào Base Model...")
        self.model = PeftModel.from_pretrained(self.model, adapter_path)
        print("✅ Đã chuyển đổi thành mô hình Fine-tuned!")

    def generate(self, question, context=""):
        prompt = f"Dựa vào thông tin cung cấp dưới đây, hãy trả lời câu hỏi một cách chính xác.\n\nThông tin:\n{context}\n\nCâu hỏi: {question}\n\nTrả lời:" if context else f"Câu hỏi: {question}\n\nTrả lời:"
            
        inputs = self.tokenizer(prompt, return_tensors="pt").to("cuda")
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            outputs = self.model.generate(**inputs, max_new_tokens=200, temperature=0.2, do_sample=True, pad_token_id=self.tokenizer.eos_token_id)
            
        answer = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        return answer.split("Trả lời:")[-1].strip()

In [6]:
class MetricsEvaluator:
    def __init__(self):
        print("⏳ Đang tải công cụ đánh giá...")
        self.bleu = evaluate.load("sacrebleu")
        self.rouge = evaluate.load("rouge")
        self.bertscore = evaluate.load("bertscore")
        print("✅ Tải công cụ đánh giá xong!")

    def evaluate_answers(self, predictions, references):
        bleu_score = self.bleu.compute(predictions=predictions, references=references)['score']
        rouge_score = self.rouge.compute(predictions=predictions, references=references)['rougeL']
        
        logging.getLogger("evaluate").setLevel(logging.ERROR)
        bs_results = self.bertscore.compute(predictions=predictions, references=references, lang="vi")
        bert_f1 = sum(bs_results['f1']) / len(bs_results['f1'])

        print(f"  + BLEU:      {bleu_score:.2f}")
        print(f"  + ROUGE-L:   {rouge_score*100:.2f}")
        print(f"  + BERTScore: {bert_f1*100:.2f}")

    # Đã thêm hàm đánh giá Retrieval với logic khớp dữ liệu jsonl của bạn
    def evaluate_retrieval(self, queries, ground_truths, rag_pipeline, k=5):
        print(f"\n⏳ Đang đánh giá Retrieval (Recall@{k})...")
        hits = 0
        valid_total = 0 
        
        for query, truth in zip(queries, ground_truths):
            # Kiểm tra xem có nhãn ground_truth không
            if not truth: 
                continue
                
            valid_total += 1
            
            # Truy xuất top_k = 5 đoạn văn bản từ FAISS
            _, docs = rag_pipeline.retrieve(query, top_k=k)
            
            # Trích xuất giá trị metadata['source'] từ các documents
            retrieved_sources = [doc.metadata.get("source", "") for doc in docs]
            
            # Kiểm tra xem chuỗi 'truth' (source_title) có nằm trong danh sách 'source' của 5 chunks này không
            if any(truth == src for src in retrieved_sources if src):
                hits += 1
                
        if valid_total > 0:
            recall = hits / valid_total
            print(f"  + Tổng số câu có nhãn Ground Truth (source_title): {valid_total}")
            print(f"  + Số lần FAISS tìm trúng đích (Hits): {hits}")
            print(f"  + Recall@{k}: {recall*100:.2f}%")
            return recall
        else:
            print("  ⚠️ Không tìm thấy nhãn 'source_title' hợp lệ trong tập test. Không thể tính Recall!")
            return 0

In [7]:
flush_memory()

config = Config()
rag = RAGPipeline(config)
rag.build_vector_store()

# --- 1. FINE-TUNING ---
print("\n" + "="*60)
print("GIAI ĐOẠN 1: FINE-TUNING")
print("="*60)
train_data = DataManager.load_qa_data(config.TRAIN_QA_PATH)
llm_train = LLMManager(config)
llm_train.finetune(train_data)

# Giải phóng VRAM ngay sau khi train xong
del llm_train
flush_memory()

# --- CHUẨN BỊ DATA TEST ---
test_dataset = load_dataset("json", data_files={"test": config.TEST_QA_PATH})['test']
num_eval = min(50, len(test_dataset))
test_questions = test_dataset['instruction'][:num_eval]
references = test_dataset['output'][:num_eval]

# CẬP NHẬT TẠI ĐÂY: Trích xuất trường `source_title` làm Ground Truth cho hệ thống RAG
if 'source_title' in test_dataset.column_names:
    ground_truths = test_dataset['source_title'][:num_eval]
else:
    ground_truths = [None] * num_eval

results_A, results_B, results_C, results_D = [], [], [], []

# --- 2. BASE MODEL ---
print("\n" + "="*60)
print("GIAI ĐOẠN 2: CHẠY BASE MODEL")
print("="*60)
llm_base = LLMManager(config)
for q in tqdm(test_questions, desc="Cấu hình A & B"):
    results_A.append(llm_base.generate(q, context=""))
    context, _ = rag.retrieve(q)
    results_B.append(llm_base.generate(q, context=context))

del llm_base
flush_memory()

# --- 3. FINE-TUNED MODEL ---
print("\n" + "="*60)
print("GIAI ĐOẠN 3: CHẠY FINE-TUNED MODEL")
print("="*60)
llm_ft = LLMManager(config)
llm_ft.load_adapter(config.OUTPUT_DIR) 
for q in tqdm(test_questions, desc="Cấu hình C & D"):
    results_C.append(llm_ft.generate(q, context=""))
    context, _ = rag.retrieve(q)
    results_D.append(llm_ft.generate(q, context=context))

del llm_ft
flush_memory()

# --- 4. TÍNH ĐIỂM & LƯU BẰNG PANDAS MẶC ĐỊNH ---
print("\n" + "="*60)
print("BẢNG TỔNG HỢP ĐIỂM SỐ ĐÁNH GIÁ")
print("="*60)
evaluator = MetricsEvaluator()

print("\n🔹 Cấu hình A (Base Model - Không RAG):")
evaluator.evaluate_answers(results_A, references)

print("\n🔹 Cấu hình B (Base Model - Có RAG):")
evaluator.evaluate_answers(results_B, references)

print("\n🔹 Cấu hình C (Fine-tuned Model - Không RAG):")
evaluator.evaluate_answers(results_C, references)

print("\n🔹 Cấu hình D (Fine-tuned Model - Có RAG):")
evaluator.evaluate_answers(results_D, references)

# KIỂM TRA HIỆU SUẤT TRUY XUẤT RAG (Recall@5)
print("\n" + "="*60)
print("ĐÁNH GIÁ HỆ THỐNG TRUY XUẤT (RETRIEVAL)")
print("="*60)
evaluator.evaluate_retrieval(test_questions, ground_truths, rag, k=5)

df = pd.DataFrame({
    "Câu hỏi": test_questions,
    "Đáp án chuẩn": references,
    "A (Base - no RAG)": results_A,
    "B (Base + RAG)": results_B,
    "C (FT - no RAG)": results_C,
    "D (FT + RAG)": results_D
})
df.to_csv("/kaggle/working/ket_qua_so_sanh_4_cau_hinh.csv", index=False)
print("\n✅ Đã lưu kết quả DataFrame bằng thư viện Pandas gốc của Kaggle!")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/752 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  + BLEU:      8.75
  + ROUGE-L:   34.90
  + BERTScore: 71.23

🔹 Cấu hình B (Base Model - Có RAG):
  + BLEU:      24.67
  + ROUGE-L:   47.51
  + BERTScore: 78.41

🔹 Cấu hình C (Fine-tuned Model - Không RAG):
  + BLEU:      15.96
  + ROUGE-L:   41.28
  + BERTScore: 76.61

🔹 Cấu hình D (Fine-tuned Model - Có RAG):
  + BLEU:      23.89
  + ROUGE-L:   48.46
  + BERTScore: 79.07

ĐÁNH GIÁ HỆ THỐNG TRUY XUẤT (RETRIEVAL)

⏳ Đang đánh giá Retrieval (Recall@5)...
  + Tổng số câu có nhãn Ground Truth (source_title): 50
  + Số lần FAISS tìm trúng đích (Hits): 20
  + Recall@5: 40.00%

✅ Đã lưu kết quả DataFrame bằng thư viện Pandas gốc của Kaggle!
